In [1]:
#Sistema Lorenz sin ruido y con todas las observaciones de las variables disponibles
#Es díficil porque a SINDy-KAN le cuesta hacer multiplicaciones
#Voy a aplicar Direct Sindy-KAN: las aristas de la red son directamente las funciones simbólicas del diccionario
#Los únicos parámetros que hay que aprender son los coeficientes de Xi y Lambda_shadow

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from scipy.integrate import odeint #soluciona EDOS

# Numeros aleatorios
torch.manual_seed(32)
np.random.seed(32)

In [3]:
#Generar datos reales
def lorenz_real(X, t):
    x, y, z = X
    sigma, rho, beta = 10.0, 28.0, 8.0/3.0
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return [dx, dy, dz]

# Generamos 2000 instantes de tiempo entre t=0 y t=10
t = np.linspace(0, 10, 2000)
X_inicial = [-8.0, 8.0, 27.0]

# Resolvemos numéricamente para obtener los estados X y las derivadas verdaderas X_dot
X_data_np = odeint(lorenz_real, X_inicial, t) #[x y z]
X_dot_data_np = np.array([lorenz_real(x_vec, 0) for x_vec in X_data_np])

# Convertimos a tensores de PyTorch porque si no las cosas ni siquiera compilan
X_tensor = torch.tensor(X_data_np, dtype=torch.float32)
X_dot_true = torch.tensor(X_dot_data_np, dtype=torch.float32)
print(f"Forma de X tensor:",X_tensor.shape)

Forma de X tensor: torch.Size([2000, 3])


In [4]:
# Construimos polinomios de grado 1 y 2 para las variables x, y, z
def build_theta_multivar(X):
    x = X[:, 0:1] # Columna de x. El 0:1 mantiene x como un vector columna, no como una simple lista
    y = X[:, 1:2] # Columna de y
    z = X[:, 2:3] # Columna de z
                    #Tmb funcionaría x = X[:, 0].unsqueeze(1)
    
    return torch.cat([
        torch.ones_like(x),          # Constante 1
        x, y, z,                     # Lineales
        x**2, y**2, z**2,            # Cuadrados 
        x*y, x*z, y*z                # Términos cruzados multiplicativos
    ], dim=1)

Theta = build_theta_multivar(X_tensor)
num_features = Theta.shape[1] # 10 características candidatas. FILAS DE THETA
num_equations = 3             # 3 ecuaciones a descubrir (dx, dy, dz). COLUMNAS DE THETA

In [5]:
# PARÁMETROS DIRECT SINDy-KAN (ahora son matrices)
# Xi y Lambda_shadow ahora son matrices de tamaño (10 filas x 3 columnas) en vez de un vector columna como en Ejemplo1_KAN
Xi = nn.Parameter(torch.randn(num_features, num_equations) * 0.01)
Lambda_shadow = nn.Parameter(torch.randn(num_features, num_equations) * 0.01)

In [6]:
# ENTRENAMIENTO 
optimizer = optim.Adam([Xi, Lambda_shadow], lr=0.005)

lam_S = 1.0      
lam_L = 1.0      
lam_1 = 0.5     
lam_2 = 1.0      

iteraciones = 15000   # Más iteraciones al ser un sistema complejo en 3D

print("Iniciando entrenamiento ADAM para el atractor de Lorenz...\n")
for k in range(iteraciones):
    optimizer.zero_grad()
    
    # Predicciones de la matriz de derivadas (2000x10 @ 10x3 = 2000x3)
    X_dot_pred_xi = Theta @ Xi
    X_dot_pred_lam = Theta @ Lambda_shadow
    
    # Loss function
    L_S = torch.mean((X_dot_true - X_dot_pred_xi)**2)
    L_L = torch.mean((X_dot_true - X_dot_pred_lam)**2)
    L_1 = torch.norm(Lambda_shadow, p=1)              
    L_2 = torch.mean((Lambda_shadow - Xi)**2)          
    
    loss = lam_S * L_S + lam_L * L_L + lam_1 * L_1 + lam_2 * L_2
    loss.backward()
    optimizer.step()
    
    if k % 3000 == 0:
        print(f"Iter {k} | Loss: {loss.item():.4f} | Norma L1: {L_1.item():.4f}")

Iniciando entrenamiento ADAM para el atractor de Lorenz...

Iter 0 | Loss: 8373.1143 | Norma L1: 0.2408
Iter 3000 | Loss: 102.2932 | Norma L1: 29.1099
Iter 6000 | Loss: 57.4939 | Norma L1: 40.4395
Iter 9000 | Loss: 29.4784 | Norma L1: 44.9111
Iter 12000 | Loss: 24.7608 | Norma L1: 46.4429


In [7]:
#UMBRAL SECUENCIAL 
Xi_final = Xi.detach().clone()
umbral = 0.5  # Borramos todo coeficiente menor a 0.5
Xi_final[Xi_final.abs() < umbral] = 0.0

In [8]:
# RESULTADOS
print("\n--- SISTEMA DE ECUACIONES DESCUBIERTO ---")
features_names = ["1", "x", "y", "z", "x^2", "y^2", "z^2", "xy", "xz", "yz"]
vars_names = ["dx/dt", "dy/dt", "dz/dt"]

for k in range(num_equations):
    ecuacion = []
    for i in range(num_features):
        coef = Xi_final[i, k].item()
        if coef != 0:
            ecuacion.append(f"{coef:+.4f}*{features_names[i]}")
            
    if not ecuacion:
        print(f"{vars_names[k]} = 0")
    else:
        print(f"{vars_names[k]} = " + " ".join(ecuacion))

print("\n--- FÍSICA REAL ESPERADA ---")
print("dx/dt = -10.0000*x +10.0000*y")
print("dy/dt = +28.0000*x -1.0000*y -1.0000*xz")
print("dz/dt = -2.6667*z +1.0000*xy")


--- SISTEMA DE ECUACIONES DESCUBIERTO ---
dx/dt = -9.1184*x +9.4567*y
dy/dt = +27.4781*x -0.6793*y -0.9856*xz
dz/dt = -2.6652*z +0.9998*xy

--- FÍSICA REAL ESPERADA ---
dx/dt = -10.0000*x +10.0000*y
dy/dt = +28.0000*x -1.0000*y -1.0000*xz
dz/dt = -2.6667*z +1.0000*xy
